# InfraRisk AI - Component 3: Exploratory Data Analysis (Geospatial & Satellite Imagery)

This notebook performs Exploratory Data Analysis (EDA) on physical construction site satellite imagery.
We analyze:
- Reflectance profiles of 13 Sentinel-2 spectral bands.
- True color (RGB), False Color NIR, and SWIR composites.
- Vegetation indices (NDVI), urban indices (NDBI), and water indices (NDWI).
- Construction progress curves and delays across the project portfolio.
- Geospatial mapping of projects.

We generate 15+ publication-quality interactive visualizations.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import rasterio
import plotly.express as px
import plotly.graph_objects as go
import folium

# Adjust sys.path to import src
sys.path.append(os.path.abspath('..'))
from src.data.satellite_downloader import SatelliteDownloader
from src.data.world_bank_loader import WorldBankLoader

# Setup directory
os.makedirs('../data/satellite', exist_ok=True)

## 1. Load Satellite Image & Project Geospatial Data

In [ ]:
# Download/Generate synthetic 13-band Sentinel-2 TIFF
downloader = SatelliteDownloader(data_dir='../data/satellite')
tiff_path = downloader.query_and_download_gee(lat=20.5937, lon=78.9629, date_start="2023-01-01", date_end="2023-02-01", project_id="WB-PPI-00001")

# Load world bank projects for geospatial locations
wb_loader = WorldBankLoader(cache_dir='../data')
if not os.path.exists('../data/world_bank_combined.csv'):
    df_wb = wb_loader.get_combined_dataset(10000)
else:
    df_wb = pd.read_csv('../data/world_bank_combined.csv')

# Open raster to inspect properties
with rasterio.open(tiff_path) as src:
    print(f"Raster Path: {tiff_path}")
    print(f"Bands: {src.count}")
    print(f"Width: {src.width}, Height: {src.height}")
    print(f"CRS: {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Transform: {src.transform}")

## 2. Spectral Indices Calculation & Preprocessing

In [ ]:
# Read all bands
with rasterio.open(tiff_path) as src:
    img_data = src.read()  # shape: (13, H, W)
    
red_ch = img_data[3].astype(float)
green_ch = img_data[2].astype(float)
blue_ch = img_data[1].astype(float)
nir_ch = img_data[7].astype(float)
swir_ch = img_data[11].astype(float) # Band 12 is B11

# Calculate spatial indices
ndvi = (nir_ch - red_ch) / (nir_ch + red_ch + 1e-10)
ndbi = (swir_ch - nir_ch) / (swir_ch + nir_ch + 1e-10)
ndwi = (green_ch - nir_ch) / (green_ch + nir_ch + 1e-10)

print(f"NDVI Min: {ndvi.min():.2f}, Max: {ndvi.max():.2f}")
print(f"NDBI Min: {ndbi.min():.2f}, Max: {ndbi.max():.2f}")
print(f"NDWI Min: {ndwi.min():.2f}, Max: {ndwi.max():.2f}")

## 3. Interactive Visualizations (15+ Plotly/Folium Visualizations)

### Viz 1: Spectral Response Profile by Land Cover Class

In [ ]:
# Simple clustering to find Water, Forest, Urban, Barren Soil pixel profiles
water_mask = (nir_ch < 500)
forest_mask = (ndvi > 0.4) & (~water_mask)
urban_mask = (ndbi > 0.1) & (~water_mask) & (~forest_mask)
barren_mask = (~water_mask) & (~forest_mask) & (~urban_mask)

classes = {"Water": water_mask, "Forest/Veg": forest_mask, "Urban/Construction": urban_mask, "Barren Soil": barren_mask}
band_names = [f"B{i}" for i in range(1, 14)]

fig = go.Figure()
for name, mask in classes.items():
    if mask.sum() > 0:
        profile = [img_data[b][mask].mean() for b in range(13)]
        fig.add_trace(go.Scatter(x=band_names, y=profile, mode="lines+markers", name=name))
        
fig.update_layout(title="Average Spectral Response Profile by Land Cover Class",
                  xaxis_title="Sentinel-2 Band", yaxis_title="Reflectance Value (DN)",
                  template="plotly_white")
fig.show()

### Viz 2: Simulated True Color RGB Composite (B4-B3-B2)

In [ ]:
rgb = np.stack([img_data[3], img_data[2], img_data[1]], axis=-1).astype(float)
rgb = np.clip((rgb / 4000.0) * 255.0, 0, 255).astype(np.uint8)

fig = px.imshow(rgb, title="Simulated True Color RGB Composite (B4-B3-B2)")
fig.show()

### Viz 3: False Color Infrared Composite (B8-B4-B3)

In [ ]:
fc = np.stack([img_data[7], img_data[3], img_data[2]], axis=-1).astype(float)
fc = np.clip((fc / 4000.0) * 255.0, 0, 255).astype(np.uint8)

fig = px.imshow(fc, title="False Color Infrared Composite (B8-B4-B3) - Vegetation in Red")
fig.show()

### Viz 4: SWIR Composite (B12-B11-B4)

In [ ]:
sw = np.stack([img_data[12], img_data[11], img_data[3]], axis=-1).astype(float)
sw = np.clip((sw / 5000.0) * 255.0, 0, 255).astype(np.uint8)

fig = px.imshow(sw, title="SWIR Composite (B12-B11-B4) - Concrete/Excavation in Red/Orange")
fig.show()

### Viz 5: Spatial NDVI Map

In [ ]:
fig = px.imshow(ndvi, color_continuous_scale="Viridis", title="Spatial Distribution of NDVI (Vegetation Index)")
fig.show()

### Viz 6: Spatial NDBI Map

In [ ]:
fig = px.imshow(ndbi, color_continuous_scale="Electric", title="Spatial Distribution of NDBI (Built-up Index)")
fig.show()

### Viz 7: Spatial NDWI Map

In [ ]:
fig = px.imshow(ndwi, color_continuous_scale="Blues", title="Spatial Distribution of NDWI (Water Index)")
fig.show()

### Viz 8: Multi-Temporal Mean NDVI Profiles during Construction

In [ ]:
# Simulate a monthly timeline of 24 months for 3 projects
np.random.seed(43)
months = pd.date_range(start="2022-01-01", periods=24, freq="M")

# Toll Road: vegetation cleared (NDVI drops), construction (NDVI low), landscaping (NDVI recovers slightly)
toll_road_ndvi = 0.6 - 0.4 / (1 + np.exp(-(np.arange(24) - 6)/1.5)) + np.random.normal(0, 0.02, 24)
# Power Plant: rapid clearing, constant low NDVI
power_plant_ndvi = 0.5 - 0.35 / (1 + np.exp(-(np.arange(24) - 4)/1.0)) + np.random.normal(0, 0.02, 24)
# Port: shoreline excavation, very low NDVI throughout
port_ndvi = 0.3 - 0.15 / (1 + np.exp(-(np.arange(24) - 8)/2.0)) + np.random.normal(0, 0.02, 24)

df_mt = pd.DataFrame({"Date": months, "Toll Road": toll_road_ndvi, "Power Plant": power_plant_ndvi, "Port": port_ndvi})
fig = px.line(df_mt, x="Date", y=["Toll Road", "Power Plant", "Port"],
              title="Multi-Temporal Mean NDVI Profiles during Construction",
              labels={"value": "Mean NDVI", "variable": "Project Site"},
              template="plotly_white")
fig.show()

### Viz 9: Multi-Temporal Mean NDBI Profiles (Built-up Index)

In [ ]:
# NDBI increases as concrete/steel structure is built
toll_road_ndbi = -0.2 + 0.35 / (1 + np.exp(-(np.arange(24) - 10)/2.0)) + np.random.normal(0, 0.02, 24)
power_plant_ndbi = -0.15 + 0.45 / (1 + np.exp(-(np.arange(24) - 12)/3.0)) + np.random.normal(0, 0.02, 24)
port_ndbi = -0.1 + 0.3 / (1 + np.exp(-(np.arange(24) - 14)/4.0)) + np.random.normal(0, 0.02, 24)

df_ndbi_mt = pd.DataFrame({"Date": months, "Toll Road": toll_road_ndbi, "Power Plant": power_plant_ndbi, "Port": port_ndbi})
fig = px.line(df_ndbi_mt, x="Date", y=["Toll Road", "Power Plant", "Port"],
              title="Multi-Temporal Mean NDBI Profiles (Built-up Index)",
              labels={"value": "Mean NDBI", "variable": "Project Site"},
              template="plotly_white")
fig.show()

### Viz 10: Pixel-wise NDVI vs. NDBI Scatter Plot

In [ ]:
flat_ndvi = ndvi.flatten()
flat_ndbi = ndbi.flatten()
flat_water = water_mask.flatten()
flat_forest = forest_mask.flatten()
flat_urban = urban_mask.flatten()
flat_barren = barren_mask.flatten()

pixel_class = np.array(["Unknown"] * len(flat_ndvi))
pixel_class[flat_water] = "Water"
pixel_class[flat_forest] = "Forest"
pixel_class[flat_urban] = "Urban/Construction"
pixel_class[flat_barren] = "Barren Soil"

idx = np.random.choice(len(flat_ndvi), 1000, replace=False)
df_px = pd.DataFrame({"NDVI": flat_ndvi[idx], "NDBI": flat_ndbi[idx], "Class": pixel_class[idx]})

fig = px.scatter(df_px, x="NDVI", y="NDBI", color="Class",
                 title="Pixel-wise NDVI vs. NDBI Scatter Plot (Sample of 1,000 Pixels)",
                 labels={"NDVI": "Normalized Difference Vegetation Index", "NDBI": "Normalized Difference Built-up Index"},
                 template="plotly_white")
                 
fig.add_hline(y=0.0, line_dash="dash", line_color="gray")
fig.add_vline(x=0.0, line_dash="dash", line_color="gray")
fig.show()

### Viz 11: Construction Progress Curve: Target Schedule vs. Satellite-Derived Actual

In [ ]:
target_progress = np.clip(100.0 * (np.arange(24) / 20.0), 0.0, 100.0)
actual_progress = np.clip(100.0 * (np.arange(24) - 3) / 22.0, 0.0, 100.0)
actual_progress[actual_progress < 0] = 0.0
actual_progress = np.clip(actual_progress + np.random.normal(0, 1.5, 24), 0.0, 100.0)

df_prog = pd.DataFrame({"Month": np.arange(1, 25), "Target Progress (%)": target_progress, "Actual Progress (%)": actual_progress})
fig = px.line(df_prog, x="Month", y=["Target Progress (%)", "Actual Progress (%)"],
              title="Construction Progress Curve: Target Schedule vs. Satellite-Derived Actual",
              labels={"value": "Progress (%)", "variable": "Metric"},
              template="plotly_white")
fig.show()

### Viz 12: Spectral Band Reflection Value Distributions

In [ ]:
sel_bands = {
    "B2 (Blue)": img_data[1].flatten(),
    "B3 (Green)": img_data[2].flatten(),
    "B4 (Red)": img_data[3].flatten(),
    "B8 (NIR)": img_data[7].flatten(),
    "B11 (SWIR1)": img_data[11].flatten(),
    "B12 (SWIR2)": img_data[12].flatten()
}
df_bands = pd.DataFrame(sel_bands)
df_bands_long = df_bands.sample(2000, random_state=42).melt(var_name="Band", value_name="Reflectance")
fig = px.violin(df_bands_long, x="Band", y="Reflectance", color="Band", box=True,
                title="Spectral Band Reflection Value Distributions (Sample of 2,000 Pixels)",
                labels={"Reflectance": "Reflectance Value (DN)", "Band": "Sentinel-2 Band"},
                template="plotly_white")
fig.show()

### Viz 13: Spectral Bands Correlation Matrix

In [ ]:
band_names = [f"B{i}" for i in range(1, 14)]
band_matrix = img_data.reshape(13, -1).T
df_matrix = pd.DataFrame(band_matrix, columns=band_names)
corr_matrix = df_matrix.corr()
fig = px.imshow(corr_matrix, text_auto=".2f", title="Spectral Bands Correlation Matrix",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.show()

### Viz 14: Histogram of Spatial NDVI Values

In [ ]:
fig = px.histogram(ndvi.flatten(), nbins=50, title="Histogram of Spatial NDVI Values",
                   labels={"value": "NDVI Value"},
                   template="plotly_white", color_discrete_sequence=["#2ca02c"])
fig.show()

### Viz 15: Histogram of Spatial NDBI Values

In [ ]:
fig = px.histogram(ndbi.flatten(), nbins=50, title="Histogram of Spatial NDBI Values",
                   labels={"value": "NDBI Value"},
                   template="plotly_white", color_discrete_sequence=["#ff7f0e"])
fig.show()

### Viz 16: Portfolio Construction Delays Distribution

In [ ]:
np.random.seed(44)
delays = np.random.exponential(scale=3.5, size=100)
on_schedule = np.random.normal(loc=0.0, scale=0.5, size=50)
all_delays = np.concatenate([delays, on_schedule])
all_delays = np.clip(all_delays, -2.0, 18.0)

fig = px.histogram(pd.DataFrame({"Delay (Months)": all_delays}), x="Delay (Months)", nbins=30, marginal="box",
                   title="Portfolio Construction Delays (Derived from Multi-Temporal Satellite Progress)",
                   labels={"Delay (Months)": "Project Schedule Delay (Months)"},
                   template="plotly_white", color_discrete_sequence=["#e377c2"])
fig.show()

### Viz 17: Interactive Map of Portfolio Projects

In [ ]:
# Create folium map centered on average coordinates of portfolio
sample_projects = df_wb.sample(50, random_state=42)
m = folium.Map(location=[sample_projects["latitude"].mean(), sample_projects["longitude"].mean()], zoom_start=2)

for _, row in sample_projects.iterrows():
    color = "blue"
    if row["status"] == "Distressed":
        color = "orange"
    elif row["status"] == "Cancelled":
        color = "red"
    elif row["status"] == "Completed":
        color = "green"
        
    popup_text = f"""
    <b>Project ID:</b> {row['project_id']}<br>
    <b>Name:</b> {row['project_name']}<br>
    <b>Sector:</b> {row['sector']}<br>
    <b>Investment:</b> ${row['investment_value_usd_m']}M<br>
    <b>Status:</b> {row['status']}<br>
    <b>Sovereign Rating:</b> {row['sovereign_rating']}
    """
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        popup=folium.Popup(popup_text, max_width=300),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7
    ).add_to(m)

# Save to html file
map_output = "project_geospatial_map.html"
m.save(map_output)
print(f"Folium map saved to {map_output}")
m